# NCAA Rowing Roster Extraction

This project is a summary of my time as a Division I rower at Duquesne University. Over four years of competing in the Atlantic 10 conference I had many meets and many more teammates. The work here uses various analysis tools (**webscraping, regex, visualizations**) to demonstrate and tell the narrative of my collegiate carrer.

In [2]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

Method to request access to the Duquesne athletics site

In [53]:
def request_access(season_years:str):
    """Make site request for webscraping data

    Args:
        season_years: desired season years in format 20XX-XX

    Returns:
        r: request message (200 indicating success)
    """
    headers = {'user-agent': 'H Valenty personal study (unc6kr@virginia.edu)'}
    r = requests.get(f"https://goduquesne.com/sports/womens-rowing/roster/{season_years}",
                     headers = headers)
    return r

Method to clean front and back end of entries

In [46]:
def remove_str_start_end(s, start, end):
    """string function to clean formatted entries"""
    return s[:start] + s[end + 1:]

Method for primary cleaning of webscraped data

In [63]:
def initial_clean(req):
    """cleaning gathered data to remove
    javascript formatting and code

    Args:
        req: request response of website data

    Returns:
        rep: cleaned data as list
    """
    # access javascript code from request to website
    roster = BeautifulSoup(req.text, 'html').find_all('script', type=True)[0]
    # format javascript as string
    str_roster = str(roster)
    # split string to isolate each rower
    roster_list = str_roster.split("Person")[1:]
    # clean formatted text from each rower
    clean = [a.strip('",\"@context":\"http://schema.org\",') for a in roster_list ]
    # call method to further clean start and end of entries
    rep = [i.strip(remove_str_start_end(clean[0], 31, -9)) for i in clean]

    return rep

Method to transform data into formatted dataframe

In [43]:
def dataframe_format(rep):
    """alter list data to more workable format as dataframe
    Args:
        rep: cleaned data in list format

    Returns:
        duq_roster: pandas dataframe of cleaned data
    """
    # fully split text into list of lists for rower entries
    hold = [rep[i].split(',') for i in range(len(rep))]
    # format into pandas dataframre and selection columns of interest
    duq_roster = pd.DataFrame(hold)[[0,3,4,5]]
    # rename columns 
    replace_map = {0:'photo_url',
              3:'athlete',
              4:'gender',
              5:'page_url'}

    duq_roster = duq_roster.rename(replace_map, axis = 1)
    # drop nulls from dataframe
    duq_roster = duq_roster.drop(duq_roster[duq_roster.photo_url == 'null'].index).reset_index(drop=True)

    return duq_roster

End process cleaning for formatted dataframe

In [52]:
def final_clean(duq_roster):
    """combination of string regex for final cleaning"""
    duq_roster['photo_url'] = [duq_roster.photo_url[i].replace('url":"', '') for i in range(len(duq_roster))]
    duq_roster.athlete = [duq_roster.athlete[i].replace('"name":"', '').strip('"') for i in range(len(duq_roster))]
    duq_roster.gender = [duq_roster.gender[i].replace('"gender":"', '').strip('"') for i in range(len(duq_roster))]
    duq_roster.page_url = [duq_roster.page_url[i].replace('"url":"', '') for i in range(len(duq_roster))]

    return duq_roster

Execute methods for 4 seasons of collegiate career

In [69]:
seasons = ['2020-21','2021-22','2022-23','2023-24']
season_df = {}
# iterate through all 4 seasons to generate data tables
for season in seasons:
    r = request_access(season)
    rep = initial_clean(req=r)
    duq_roster = dataframe_format(rep=rep)
    season_df["season-{0}".format(season)] = final_clean(duq_roster=duq_roster)

Show snippet of 2020-21 season roster

In [72]:
season_df['season-2020-21'].head()

,photo_url,athlete,gender,page_url
0,https://goduquesne.com/images/2020/11/17/Bridg...,Bridget Abbott,F,https://goduquesne.com/roster.aspx?rp_id=10488
1,https://goduquesne.com/images/2019/8/1/New_D_h...,Isabella Abbott,F,https://goduquesne.com/roster.aspx?rp_id=10489
2,https://goduquesne.com/images/2020/11/17/Kathr...,Kathryn Ackerman,F,https://goduquesne.com/roster.aspx?rp_id=10474
3,https://goduquesne.com/images/2020/11/17/Kelly...,Kelly Ardrey,F,https://goduquesne.com/roster.aspx?rp_id=10482
4,https://goduquesne.com/images/2020/3/16/Asmund...,Aleiia Asmundson,F,https://goduquesne.com/roster.aspx?rp_id=10436


In [23]:
# export rosters to csv and/or SQL 

-----------

## Webscraping Lineups Extraction

In [24]:
# straight to the article link of interest
headers = {'user-agent': 'H Valenty personal study (unc6kr@virginia.edu)'}
r = requests.get("https://goduquesne.com/news/2024/5/20/womens-rowing-dukes-finish-seventh-at-atlantic-10-championship.aspx", headers = headers)
print(r)

<Response [200]>


In [25]:
a10s = BeautifulSoup(r.text, 'html')

In [166]:
lineups = a10s.find_all('td')
lineups
# yay its already in a list

[<td style="width: 158px;"><span style="color:#B22222;"><strong>1st Varsity 8+</strong></span></td>,
 <td style="width: 200px;"><span style="color:#B22222;"><strong>2nd Varsity 8+</strong></span></td>,
 <td style="width: 167px;"><span style="color:#B22222;"><strong>3rd Varsity 8+</strong></span></td>,
 <td style="width: 180px;"><span style="color:#B22222;"><strong>1st Varsity 4+</strong></span></td>,
 <td style="width: 158px;">Cox: <dfn><a href="/sports/womens-rowing/roster/meghan-mangan/12174" rel="smarttag" rev="12174">Meghan Mangan</a></dfn></td>,
 <td style="width: 200px;">Cox: <dfn><a href="/sports/womens-rowing/roster/catherine-egan/12263" rel="smarttag" rev="12263">Catherine Egan</a></dfn></td>,
 <td style="width: 167px;">Cox: <dfn><a href="/sports/womens-rowing/roster/rory-brouillard/12162" rel="smarttag" rev="12162">Rory Brouillard</a></dfn></td>,
 <td style="width: 180px;">Cox: <dfn><a href="/sports/womens-rowing/roster/paige-engel/12196" rel="smarttag" rev="12196">Paige Enge

In [167]:
# make it a string and go to work
str_lineups = str(lineups)
lineups_list = str_lineups.split(',')
lineups_list

['[<td style="width: 158px;"><span style="color:#B22222;"><strong>1st\xa0Varsity 8+</strong></span></td>',
 ' <td style="width: 200px;"><span style="color:#B22222;"><strong>2nd\xa0Varsity 8+</strong></span></td>',
 ' <td style="width: 167px;"><span style="color:#B22222;"><strong>3rd Varsity 8+</strong></span></td>',
 ' <td style="width: 180px;"><span style="color:#B22222;"><strong>1st Varsity 4+</strong></span></td>',
 ' <td style="width: 158px;">Cox: <dfn><a href="/sports/womens-rowing/roster/meghan-mangan/12174" rel="smarttag" rev="12174">Meghan Mangan</a></dfn></td>',
 ' <td style="width: 200px;">Cox: <dfn><a href="/sports/womens-rowing/roster/catherine-egan/12263" rel="smarttag" rev="12263">Catherine Egan</a></dfn></td>',
 ' <td style="width: 167px;">Cox: <dfn><a href="/sports/womens-rowing/roster/rory-brouillard/12162" rel="smarttag" rev="12162">Rory Brouillard</a></dfn></td>',
 ' <td style="width: 180px;">Cox: <dfn><a href="/sports/womens-rowing/roster/paige-engel/12196" rel="sma

In [168]:
# boat titles
ch = '<strong>'
pattern = ".*" + ch
# remove everything prior to and including the character defined
lineups_list[0:4] = [re.sub(pattern, '', lineups_list[i]) for i in range(4)]
len(lineups_list)

44

In [169]:
# lineups within boats
ch = ';">'
pattern = ".*" + ch
lineups_list[4:] = [re.sub(pattern, '', lineups_list[i+4]) for i in range(39)]

In [170]:
# keep cleaning - remove random table formatting characters
lineups_list = [i for i in lineups_list if i != '</td>']
lineups_list

['1st\xa0Varsity 8+</strong></span></td>',
 '2nd\xa0Varsity 8+</strong></span></td>',
 '3rd Varsity 8+</strong></span></td>',
 '1st Varsity 4+</strong></span></td>',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/meghan-mangan/12174" rel="smarttag" rev="12174">Meghan Mangan</a></dfn></td>',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/catherine-egan/12263" rel="smarttag" rev="12263">Catherine Egan</a></dfn></td>',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/rory-brouillard/12162" rel="smarttag" rev="12162">Rory Brouillard</a></dfn></td>',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/paige-engel/12196" rel="smarttag" rev="12196">Paige Engel</a></dfn></td>',
 'Stroke: <dfn><a href="/sports/womens-rowing/roster/bridget-abbott/12128" rel="smarttag" rev="12128">Bridget Abbott</a></dfn></td>',
 'Stroke: <dfn><a href="/sports/womens-rowing/roster/samantha-szlachta/12257" rel="smarttag" rev="12257">Samantha Szlachta</a></dfn></td>',
 'Stroke: <dfn><a href="/sports/womens-row

In [171]:
# removing more table formatting html language
lineups_list = [lineups_list[i].replace('</a></dfn></td>', '') for i in range(len(lineups_list))]
lineups_list = [lineups_list[i].replace('</strong></span></td>', '') for i in range(len(lineups_list))]
lineups_list = [lineups_list[i].replace('\xa0', ' ') for i in range(len(lineups_list))]
lineups_list

['1st Varsity 8+',
 '2nd Varsity 8+',
 '3rd Varsity 8+',
 '1st Varsity 4+',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/meghan-mangan/12174" rel="smarttag" rev="12174">Meghan Mangan',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/catherine-egan/12263" rel="smarttag" rev="12263">Catherine Egan',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/rory-brouillard/12162" rel="smarttag" rev="12162">Rory Brouillard',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/paige-engel/12196" rel="smarttag" rev="12196">Paige Engel',
 'Stroke: <dfn><a href="/sports/womens-rowing/roster/bridget-abbott/12128" rel="smarttag" rev="12128">Bridget Abbott',
 'Stroke: <dfn><a href="/sports/womens-rowing/roster/samantha-szlachta/12257" rel="smarttag" rev="12257">Samantha Szlachta',
 'Stroke: <dfn><a href="/sports/womens-rowing/roster/kelly-ardrey/12131" rel="smarttag" rev="12131">Kelly Ardrey',
 'Stroke: <dfn><a href="/sports/womens-rowing/roster/nora-grace-foglia/12256" rel="smarttag" rev="12256">No

In [172]:
#lineups separated by boat
V8_1 = lineups_list[0:25:4] + lineups_list[27::3]
V8_2 = lineups_list[1:26:4] + lineups_list[28::3]
V8_3 = lineups_list[2:27:4] + lineups_list[29::3]
V4_1 = lineups_list[3:24:4]

In [173]:
# format to dataframe
varsity8 = pd.DataFrame(V8_1)
varsity8 = varisty8.drop(index=0).reset_index(drop=True)
varsity8

,1st Varsity 8+
0,"Cox: <dfn><a href=""/sports/womens-rowing/roste..."
1,"Stroke: <dfn><a href=""/sports/womens-rowing/ro..."
2,"7: <dfn><a href=""/sports/womens-rowing/roster/..."
3,"6: <dfn><a href=""/sports/womens-rowing/roster/..."
4,"5: <dfn><a href=""/sports/womens-rowing/roster/..."
5,"4: <dfn><a href=""/sports/womens-rowing/roster/..."
6,"3: <dfn><a href=""/sports/womens-rowing/roster/..."
7,"2: <dfn><a href=""/sports/womens-rowing/roster/..."
8,"Bow: <dfn><a href=""/sports/womens-rowing/roste..."


In [174]:
# split to keep seat position
varsity8[['Seat', 'Athlete']] = varsity8['1st Varsity 8+'].str.split(':', n=1, expand=True)

In [175]:
# remove original column
varsity8 = varsity8.drop('1st Varsity 8+', axis=1)

In [176]:
# final clean dropping all characters besides athlete name
ch = '">'
pattern = ".*" + ch
varsity8['Athlete'] = [re.sub(pattern, '', varsity8['Athlete'].iloc[i]) for i in range(len(varsity8))]

In [177]:
varsity8

,Seat,Athlete
0,Cox,Meghan Mangan
1,Stroke,Bridget Abbott
2,7,Julia Casey
3,6,Hannah Valenty
4,5,Cherise Dicke
5,4,Megan McMahon
6,3,Natalie Hesch
7,2,Britta Wheeler
8,Bow,Isabella Abbott


In [178]:
# repeat for 2V8

# format to dataframe
second_v8 = pd.DataFrame(V8_2)
second_v8 = second_v8.drop(index=0).reset_index(drop=True)
# split to keep seat position
second_v8[['Seat', 'Athlete']] = second_v8[0].str.split(':', n=1, expand=True)
# remove original column
second_v8 = second_v8.drop(0, axis=1)
# final clean dropping all characters besides athlete name
ch = '">'
pattern = ".*" + ch
second_v8['Athlete'] = [re.sub(pattern, '', second_v8['Athlete'].iloc[i]) for i in range(len(second_v8))]
second_v8

,Seat,Athlete
0,Cox,Catherine Egan
1,Stroke,Samantha Szlachta
2,7,Olivia Sullivan
3,6,Danielle Smith
4,5,Ella Hinchey
5,4,Nya Muffoletto
6,3,Kathryn Ackerman
7,2,Megan Elliott
8,Bow,Grace Kennevan


In [179]:
# repeat for 3V8

# format to dataframe
third_v8 = pd.DataFrame(V8_3)
third_v8 = third_v8.drop(index=0).reset_index(drop=True)
# split to keep seat position
third_v8[['Seat', 'Athlete']] = third_v8[0].str.split(':', n=1, expand=True)
# remove original column
third_v8 = third_v8.drop(0, axis=1)
# final clean dropping all characters besides athlete name
ch = '">'
pattern = ".*" + ch
third_v8['Athlete'] = [re.sub(pattern, '', third_v8['Athlete'].iloc[i]) for i in range(len(third_v8))]
third_v8

,Seat,Athlete
0,Cox,Rory Brouillard
1,Stroke,Kelly Ardrey
2,7,Chloe Ernst
3,6,Fiona Riordan
4,5,Lauren Lamer
5,4,Caitlin DeStefano
6,3,Maggie Ray
7,2,Alice Benavides
8,Bow,Emma Mills


In [180]:
# repeat for 1V4

# format to dataframe
first_v4 = pd.DataFrame(V4_1)
first_v4 = first_v4.drop(index=0).reset_index(drop=True)
# split to keep seat position
first_v4[['Seat', 'Athlete']] = first_v4[0].str.split(':', n=1, expand=True)
# remove original column
first_v4 = first_v4.drop(0, axis=1)
# final clean dropping all characters besides athlete name
ch = '">'
pattern = ".*" + ch
first_v4['Athlete'] = [re.sub(pattern, '', first_v4['Athlete'].iloc[i]) for i in range(len(first_v4))]
first_v4

,Seat,Athlete
0,Cox,Paige Engel
1,Stroke,Nora Grace Foglia
2,3,Kyra Tziovannis
3,2,Presley Oliphant
4,Bow,Nicole Woroszylo


## Replicate lineups extraction with GMU Invite

In [181]:
# straight to the article link of interest
headers = {'user-agent': 'H Valenty personal study (unc6kr@virginia.edu)'}
r = requests.get("https://goduquesne.com/news/2024/4/24/womens-rowing-full-complement-of-boats-for-dukes-compete-at-mason-invite.aspx", 
                 headers = headers)
print(r)

<Response [200]>


In [216]:
gmu = BeautifulSoup(r.text, 'html')
lineups = gmu.find_all('td')

In [288]:
# make it a string and go to work
str_lineups = str(lineups)
lineups_list = str_lineups.split(',')

In [289]:
# boat titles
ch = '<strong>'
pattern = ".*" + ch
# remove everything prior to and including the character defined
lineups_list[0:6] = [re.sub(pattern, '', lineups_list[i]) for i in range(6)]
len(lineups_list)

66

In [290]:
# lineups within boats
ch = ';">'
pattern = ".*" + ch
lineups_list[6:] = [re.sub(pattern, '', lineups_list[i+6]) for i in range(59)]
# keep cleaning - remove random table formatting characters
lineups_list = [i for i in lineups_list if i != '</td>']

In [293]:
# removing more table formatting html language
lineups_list = [lineups_list[i].replace('</a></dfn></td>', '') for i in range(len(lineups_list))]
lineups_list = [lineups_list[i].replace('</strong></span></td>', '') for i in range(len(lineups_list))]
lineups_list = [lineups_list[i].replace('\xa0', ' ') for i in range(len(lineups_list))]
lineups_list = [lineups_list[i].replace('</a></dfn>', '') for i in range(len(lineups_list))]
lineups_list = [lineups_list[i].replace('</td>', '') for i in range(len(lineups_list))]
lineups_list

['1st Varsity 8+',
 '2nd Varsity 8+',
 '1st Varsity 4+',
 '2nd Varsity 4+',
 '3rd Varsity 8+',
 '3rd Varsity / Novice 4+',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/meghan-mangan/12174" rel="smarttag" rev="12174">Meghan Mangan',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/paige-engel/12196" rel="smarttag" rev="12196">Paige Engel',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/jessica-desaro/12259" rel="smarttag" rev="12259">Jessica DeSaro',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/rory-brouillard/12162" rel="smarttag" rev="12162">Rory Brouillard',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/julia-laurie/12173" rel="smarttag" rev="12173">Julia Laurie',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/serena-decosmo/12261" rel="smarttag" rev="12261">Serena DeCosmo (heat)',
 'Cox: <dfn><a href="/sports/womens-rowing/roster/kat-muha/12260" rel="smarttag" rev="12260">Kat Muha (final)',
 'Stroke: <dfn><a href="/sports/womens-rowing/roster/bridget-abbott/1212

In [294]:
# lineups by boat
v8_1 = lineups_list[0:7:6] + lineups_list[13:38:6] + lineups_list[40::3]
v8_2 = lineups_list[1:8:6] + lineups_list[14:39:6] + lineups_list[41::3]
v4_1 = lineups_list[2:9:6] + lineups_list[15:39:6]
v4_2 = lineups_list[3:10:6] + lineups_list[16:40:6]
v8_3 = lineups_list[4:11:6] + lineups_list[17:41:6] + lineups_list[39::3]
v4_3 = lineups_list[5:12:6] + lineups_list[12:13] + lineups_list[18:42:6]

In [295]:
# first varsity

# format to dataframe
first_v8 = pd.DataFrame(v8_1)
first_v8 = first_v8.drop(index=0).reset_index(drop=True)
# split to keep seat position
first_v8[['Seat', 'Athlete']] = first_v8[0].str.split(':', n=1, expand=True)
# remove original column
first_v8 = first_v8.drop(0, axis=1)
# final clean dropping all characters besides athlete name
ch = '">'
pattern = ".*" + ch
first_v8['Athlete'] = [re.sub(pattern, '', first_v8['Athlete'].iloc[i]) for i in range(len(first_v8))]
first_v8

,Seat,Athlete
0,Cox,Meghan Mangan
1,Stroke,Bridget Abbott
2,7,Julia Casey
3,6,Cherise Dicke
4,5,Natalie Hesch
5,4,Hannah Valenty
6,3,Samantha Szlachta
7,2,Britta Wheeler
8,Bow,Isabella Abbott


In [296]:
# repeat for 2V8

# format to dataframe
second_v8 = pd.DataFrame(v8_2)
second_v8 = second_v8.drop(index=0).reset_index(drop=True)
# split to keep seat position
second_v8[['Seat', 'Athlete']] = second_v8[0].str.split(':', n=1, expand=True)
# remove original column
second_v8 = second_v8.drop(0, axis=1)
# final clean dropping all characters besides athlete name
ch = '">'
pattern = ".*" + ch
second_v8['Athlete'] = [re.sub(pattern, '', second_v8['Athlete'].iloc[i]) for i in range(len(second_v8))]
second_v8

,Seat,Athlete
0,Cox,Paige Engel
1,Stroke,Megan McMahon
2,7,Megan Elliott
3,6,Olivia Sullivan
4,5,Kyra Tziovannis
5,4,Nya Muffoletto
6,3,Presley Oliphant
7,2,Nora Grace Foglia
8,Bow,Grace Kennevan


In [297]:
# repeat for 1V4

# format to dataframe
first_v4 = pd.DataFrame(v4_1)
first_v4 = first_v4.drop(index=0).reset_index(drop=True)
# split to keep seat position
first_v4[['Seat', 'Athlete']] = first_v4[0].str.split(':', n=1, expand=True)
# remove original column
first_v4 = first_v4.drop(0, axis=1)
# final clean dropping all characters besides athlete name
ch = '">'
pattern = ".*" + ch
first_v4['Athlete'] = [re.sub(pattern, '', first_v4['Athlete'].iloc[i]) for i in range(len(first_v4))]
first_v4

,Seat,Athlete
0,Cox,Jessica DeSaro
1,Stroke,Ella Hinchey
2,3,Kelly Ardrey
3,2,Maggie Ray
4,Bow,Eliana Meding


In [298]:
# repeat for 3V8

# format to dataframe
third_v8 = pd.DataFrame(v8_3)
third_v8 = third_v8.drop(index=0).reset_index(drop=True)
# split to keep seat position
third_v8[['Seat', 'Athlete']] = third_v8[0].str.split(':', n=1, expand=True)
# remove original column
third_v8 = third_v8.drop(0, axis=1)
# final clean dropping all characters besides athlete name
ch = '">'
pattern = ".*" + ch
third_v8['Athlete'] = [re.sub(pattern, '', third_v8['Athlete'].iloc[i]) for i in range(len(third_v8))]
third_v8

,Seat,Athlete
0,Cox,Julia Laurie
1,Stroke,Allyson Gallagher
2,7,Lauren Lamer
3,6,Caitlin DeStefano
4,5,Amelia Nicholas
5,4,Fiona Riordan
6,3,Caitlin Hahn
7,2,Mary Perez
8,Bow,Antonina D'Eramo


In [299]:
# repeat for 2V4

# format to dataframe
second_v4 = pd.DataFrame(v4_2)
second_v4 = second_v4.drop(index=0).reset_index(drop=True)
# split to keep seat position
second_v4[['Seat', 'Athlete']] = second_v4[0].str.split(':', n=1, expand=True)
# remove original column
second_v4 = second_v4.drop(0, axis=1)
# final clean dropping all characters besides athlete name
ch = '">'
pattern = ".*" + ch
second_v4['Athlete'] = [re.sub(pattern, '', second_v4['Athlete'].iloc[i]) for i in range(len(second_v4))]
second_v4

,Seat,Athlete
0,Cox,Rory Brouillard
1,Stroke,Nicole Woroszylo
2,3,Kathryn Ackerman
3,2,Alice Benavides
4,Bow,Chloe Ernst


In [300]:
# repeat for 3V4

# format to dataframe
third_v4 = pd.DataFrame(v4_3)
third_v4 = third_v4.drop(index=0).reset_index(drop=True)
# split to keep seat position
third_v4[['Seat', 'Athlete']] = third_v4[0].str.split(':', n=1, expand=True)
# remove original column
third_v4 = third_v4.drop(0, axis=1)
# final clean dropping all characters besides athlete name
ch = '">'
pattern = ".*" + ch
third_v4['Athlete'] = [re.sub(pattern, '', third_v4['Athlete'].iloc[i]) for i in range(len(third_v4))]
third_v4

,Seat,Athlete
0,Cox,Serena DeCosmo (heat)
1,Cox,Kat Muha (final)
2,Stroke,Cailyn LaRosa
3,3,Emma Mills
4,2,Odessa Baker
5,Bow,Michelle Catao
